# Toronto Police Strip Search & Mental Health Analysis**Dataset**: Toronto Police Service — Arrests & Strip Searches (2020–2021)  **Records**: 65,276 cases  **Focus**: Effects of gender, age group, year, and location on strip searches and mental health incidentsThis notebook presents a full statistical analysis pipeline: EDA → Power Analysis → T-Tests → ANOVA → ANCOVA → Logistic Regression.

## 1. Setup & Data Loading

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy import statsfrom scipy.stats import levenefrom math import sqrtimport statsmodels.api as smimport statsmodels.stats.api as smsimport statsmodels.stats.power as smpimport statsmodels.formula.api as smffrom statsmodels.formula.api import olsfrom statsmodels.stats.power import TTestIndPower, TTestPowerfrom statsmodels.graphics.factorplots import interaction_plotfrom statsmodels.stats.multicomp import pairwise_tukeyhsdfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import confusion_matrix, accuracy_score%matplotlib inlinesns.set_style("whitegrid")plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load datasetdataset = pd.read_csv('data/Arrests_and_Strip_Searches.csv')print(f"Dataset shape: {dataset.shape}")print(f"\nColumns: {list(dataset.columns)}")dataset.head()

## 2. Data Preprocessing & Aggregation

We aggregate the raw 65K records into two analysis-ready DataFrames:- **`df_mental`**: Mental instability case counts grouped by Sex, Year, and ArrestLocDiv- **`df_strip`**: Strip search counts grouped by Sex, Year, and Age Group

In [ ]:
# Aggregate mental instability cases by Sex, Year, and Locationmental_agg = dataset.groupby(['Sex', 'Arrest_Year', 'ArrestLocDiv'])['Actions_at_arrest___Mental_inst'].sum().reset_index()mental_agg.columns = ['Sex', 'Year', 'ArrestLocDiv', 'Mental_inst_count']# Filter to M and F only (exclude unknown)df_mental = mental_agg[mental_agg['Sex'].isin(['M', 'F'])].copy()df_mental['Year'] = df_mental['Year'].astype(str)print(f"Mental health aggregated rows: {len(df_mental)}")df_mental.head(10)

In [ ]:
# Aggregate strip search cases by Sex, Year, and Age Groupstrip_agg = dataset.groupby(['Sex', 'Arrest_Year', 'Age_group__at_arrest_'])['StripSearch'].sum().reset_index()strip_agg.columns = ['Sex', 'Year', 'Age_group__at_arrest_', 'StripSearch']# Filter to M and F onlydf_strip = strip_agg[strip_agg['Sex'].isin(['M', 'F'])].copy()df_strip['Year'] = df_strip['Year'].astype(str)print(f"Strip search aggregated rows: {len(df_strip)}")df_strip.head(10)

## 3. Exploratory Data Analysis

### 3.1 Mental Health Cases by Location

In [ ]:
# Mental health cases by arrest location divisionfig, ax = plt.subplots(figsize=(14, 6))df_location = dataset.groupby('ArrestLocDiv')['Actions_at_arrest___Mental_inst'].sum().sort_values(ascending=False)df_location.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')ax.set_title('Mental Instability Cases by Toronto Police Division', fontsize=14, fontweight='bold')ax.set_xlabel('Arrest Location Division')ax.set_ylabel('Total Cases')plt.tight_layout()plt.show()

In [ ]:
# Box plot: Mental health cases by division and yearfig, ax = plt.subplots(figsize=(16, 8))sns.boxplot(data=df_mental, x='ArrestLocDiv', y='Mental_inst_count', hue='Year', ax=ax)ax.set_title('Distribution of Mental Health Cases by Division and Year', fontsize=14, fontweight='bold')ax.set_xlabel('Arrest Location Division')ax.set_ylabel('Case Count')ax.set_ylim(0, 260)plt.tight_layout()plt.show()

### 3.2 Strip Searches by Gender and Age Group

In [ ]:
# Strip searches by genderfig, axes = plt.subplots(1, 2, figsize=(14, 5))dataset.groupby('Sex')['StripSearch'].sum().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')axes[0].set_title('Strip Searches by Sex', fontsize=13, fontweight='bold')axes[0].set_ylabel('Total Strip Searches')axes[0].tick_params(axis='x', rotation=0)dataset.groupby('Age_group__at_arrest_')['StripSearch'].sum().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')axes[1].set_title('Strip Searches by Age Group', fontsize=13, fontweight='bold')axes[1].set_ylabel('Total Strip Searches')axes[1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.show()

In [ ]:
# Strip searches by age group and sex (box plot)fig, ax = plt.subplots(figsize=(14, 6))sns.boxplot(data=df_strip, x='Age_group__at_arrest_', y='StripSearch', hue='Sex', ax=ax)ax.set_title('Strip Search Distribution by Age Group and Sex', fontsize=14, fontweight='bold')ax.set_xlabel('Age Group')ax.set_ylabel('Strip Search Count')plt.tight_layout()plt.show()

## 4. Power Analysis

Before conducting hypothesis tests, we perform a power analysis to determine the required sample size for detecting effects with 80% power at α = 0.05.

In [ ]:
# Power analysis for independent samples t-testpower_analysis = smp.TTestIndPower()sample_size = power_analysis.solve_power(effect_size=0.5, power=0.8, alpha=0.05)print(f"Required sample size per group (medium effect, d=0.5): {np.ceil(sample_size):.0f}")

In [ ]:
# Power curves for different effect sizeseffect_sizes = np.array([0.2, 0.5, 0.8])sample_sizes = np.array(range(10, 600, 10))fig, ax = plt.subplots(figsize=(10, 6))power_analysis.plot_power(dep_var='nobs', nobs=sample_sizes,                          effect_size=effect_sizes, alpha=0.05, ax=ax,                          title='Power of Independent Samples t-test (α = 0.05)')plt.tight_layout()plt.show()

In [ ]:
# Calculate effect size from pilot study datan1, n2 = 36, 36s1, s2 = 22, 63  # variance of samples in pilot study# Pooled standard deviation (Cohen's d)s_pooled = sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))u1, u2 = 90, 85  # means from pilotd = (u1 - u2) / s_pooledprint(f"Effect size (Cohen's d): {d:.4f}")# Required sample size for this effectn_required = power_analysis.solve_power(effect_size=d, alpha=0.05, power=0.8)print(f"Required sample size per group: {np.ceil(n_required):.0f}")# Power at current sample sizepower_at_n = TTestPower().solve_power(nobs=36, effect_size=0.5, power=None, alpha=0.05)print(f"Power at n=36 with d=0.5: {power_at_n:.3f}")

## 5. Hypothesis Testing: T-Tests

### 5.1 Mental Health Cases by Gender**H₀**: No difference between males and females in mental instability case counts.  **H₁**: Significant difference exists between males and females.

In [ ]:
# Split by gendermale_mental = df_mental[df_mental['Sex'] == 'M']['Mental_inst_count']female_mental = df_mental[df_mental['Sex'] == 'F']['Mental_inst_count']print("Descriptive Statistics:")print(f"  Males   — Mean: {male_mental.mean():.1f}, SD: {male_mental.std():.1f}, n: {len(male_mental)}")print(f"  Females — Mean: {female_mental.mean():.1f}, SD: {female_mental.std():.1f}, n: {len(female_mental)}")# Welch's t-test (unequal variance)t_stat, p_val = stats.ttest_ind(male_mental, female_mental, equal_var=False)print(f"\nWelch's t-test: t = {t_stat:.4f}, p = {p_val:.4f}")# Confidence intervalcm = sms.CompareMeans(sms.DescrStatsW(male_mental), sms.DescrStatsW(female_mental))ci = cm.tconfint_diff(usevar='unequal')print(f"95% CI for difference: [{ci[0]:.2f}, {ci[1]:.2f}]")# Welch's degrees of freedomdef welch_dof(x, y):    return (x.var()/x.size + y.var()/y.size)**2 / \           ((x.var()/x.size)**2/(x.size-1) + (y.var()/y.size)**2/(y.size-1))print(f"Welch DOF: {welch_dof(male_mental, female_mental):.2f}")print(f"\n→ {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'} (α = 0.05)")

### 5.2 Strip Searches by Gender**H₀**: No significant difference in strip search counts between males and females.  **H₁**: Significant difference exists.

In [ ]:
male_strip = df_strip[df_strip['Sex'] == 'M']['StripSearch']female_strip = df_strip[df_strip['Sex'] == 'F']['StripSearch']t_stat, p_val = stats.ttest_ind(male_strip, female_strip)print(f"Two-sample t-test: t = {t_stat:.4f}, p = {p_val:.4f}")print(f"\n→ {'Reject H₀' if p_val < 0.05 else 'Fail to reject H₀'} (α = 0.05)")

## 6. Two-Way ANOVA Tests

We conduct four two-way ANOVA tests to examine main effects and interactions between categorical predictors on strip search counts and mental health case counts.

### 6.1 Strip Search ~ Sex × Age Group

In [ ]:
model1 = ols("StripSearch ~ C(Sex) + C(Age_group__at_arrest_) + C(Sex):C(Age_group__at_arrest_)", data=df_strip).fit()print(sm.stats.anova_lm(model1, typ=2))print("\n→ No significant main effects or interaction (all p > 0.05)")

### 6.2 Strip Search ~ Year × Age Group

In [ ]:
model2 = ols("StripSearch ~ C(Year) + C(Age_group__at_arrest_) + C(Year):C(Age_group__at_arrest_)", data=df_strip).fit()print(sm.stats.anova_lm(model2, typ=2))print("\n→ Year shows significant main effect; Age Group and interaction not significant")

### 6.3 Strip Search ~ Sex × Year

In [ ]:
model3 = ols("StripSearch ~ C(Sex) + C(Year) + C(Sex):C(Year)", data=df_strip).fit()print(sm.stats.anova_lm(model3, typ=2))print("\n→ Both main effects and interaction are significant (all p < 0.05)")

### 6.4 Mental Health Cases ~ Sex × ArrestLocDiv

In [ ]:
model4 = ols("Mental_inst_count ~ C(ArrestLocDiv) + C(Sex) + C(ArrestLocDiv):C(Sex)", data=df_mental).fit()print(sm.stats.anova_lm(model4, typ=2))print("\n→ All effects significant: Sex, Location, and their interaction")

## 7. Post-Hoc Analysis: Tukey HSD

Since the ANOVA for Strip Search ~ Sex × Year was significant, we perform Tukey's HSD to identify which specific group pairs differ.

In [ ]:
# Create group labels for Tukey testgroups = []values = []for sex in ['M', 'F']:    for year in ['2020', '2021']:        mask = (df_strip['Sex'] == sex) & (df_strip['Year'] == year)        vals = df_strip.loc[mask, 'StripSearch'].values        values.extend(vals)        groups.extend([f"{sex.lower()}_{year}"] * len(vals))df_tukey = pd.DataFrame({'StripSearch': values, 'Group': groups})tukey = pairwise_tukeyhsd(endog=df_tukey['StripSearch'], groups=df_tukey['Group'], alpha=0.05)print(tukey)

## 8. Interaction Plots

In [ ]:
# Interaction: Mental health cases by Location and Sexfig, axes = plt.subplots(1, 3, figsize=(20, 6))interaction_plot(df_mental['ArrestLocDiv'], df_mental['Sex'], df_mental['Mental_inst_count'].values,                 colors=['red', 'blue'], markers=['D', '^'], ms=5, ax=axes[0])axes[0].set_ylabel('Mean Case Count')axes[0].set_xlabel('Location Division')axes[0].set_title('Mental Health Cases\nby Location × Sex', fontweight='bold')# Interaction: Strip search by Age Group and Sexinteraction_plot(df_strip['Age_group__at_arrest_'], df_strip['Sex'], df_strip['StripSearch'].values,                 colors=['red', 'blue'], markers=['D', '^'], ms=5, ax=axes[1])axes[1].set_ylabel('Mean Strip Search Count')axes[1].set_xlabel('Age Group')axes[1].set_title('Strip Searches\nby Age Group × Sex', fontweight='bold')axes[1].tick_params(axis='x', rotation=45)# Interaction: Strip search by Year and Sexinteraction_plot(df_strip['Year'], df_strip['Sex'], df_strip['StripSearch'].values,                 colors=['red', 'blue'], markers=['D', '^'], ms=5, ax=axes[2])axes[2].set_ylabel('Mean Strip Search Count')axes[2].set_xlabel('Year')axes[2].set_title('Strip Searches\nby Year × Sex', fontweight='bold')plt.tight_layout()plt.show()

## 9. ANCOVA

We use Analysis of Covariance to examine the effect of Sex on outcome variables while controlling for covariates.

### 9.1 Mental Health ~ Sex, controlling for Year

In [ ]:
model_ancova1 = ols('Mental_inst_count ~ Sex + Year', data=df_mental).fit()print(model_ancova1.summary())print("\n→ Sex is significant (p = 0.005); Year is not significant (p = 0.935)")

### 9.2 Mental Health ~ Sex, controlling for Location

In [ ]:
model_ancova2 = ols('Mental_inst_count ~ Sex + ArrestLocDiv', data=df_mental).fit()print(model_ancova2.summary())print("\n→ R² = 0.806 — Location explains most variance; Sex remains significant")

### 9.3 Strip Search ~ Sex, controlling for Year

In [ ]:
model_ancova3 = ols('StripSearch ~ Sex + Year', data=df_strip).fit()print(model_ancova3.summary())print("\n→ Both Sex (p = 0.024) and Year (p = 0.007) are significant predictors")

### 9.4 Strip Search ~ Sex, controlling for Age Group

In [ ]:
model_ancova4 = ols('StripSearch ~ Sex + Age_group__at_arrest_', data=df_strip).fit()print(model_ancova4.summary())print("\n→ Sex significant (p = 0.042); Age groups not individually significant")

## 10. Logistic Regression

### 10.1 Predicting Strip Search LikelihoodWe model the probability of a strip search occurring based on arrest year, sex, and age group.

In [ ]:
# Prepare data — drop columns not used as predictorsdataset_clean = dataset.drop([    'Arrest_Month', 'EventID', 'ArrestID', 'PersonID', 'Perceived_Race',    'Youth_at_arrest__under_18_years', 'Occurrence_Category',    'SearchReason_CauseInjury', 'SearchReason_AssistEscape',    'SearchReason_PossessWeapons', 'SearchReason_PossessEvidence',    'ItemsFound', 'ObjectId',    'Actions_at_arrest___Concealed_i', 'Actions_at_arrest___Combative__',    'Actions_at_arrest___Resisted__d', 'Actions_at_arrest___Assaulted_o',    'Actions_at_arrest___Cooperative', 'Booked'], axis=1)# Model 1: Predict StripSearchX1 = dataset_clean.drop(['ArrestLocDiv', 'StripSearch', 'Actions_at_arrest___Mental_inst'], axis=1)y1 = dataset_clean['StripSearch']X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=0)formula1 = "StripSearch ~ Arrest_Year + Sex + Age_group__at_arrest_"training1 = pd.concat([X1_train, y1_train], axis=1)log_model1 = smf.logit(formula1, data=training1).fit()print(log_model1.summary())

In [ ]:
# Model 1: Evaluationpred1 = log_model1.predict(X1_test)pred1_binary = list(map(round, pred1))accuracy1 = accuracy_score(y1_test, pred1_binary)cm1 = confusion_matrix(y1_test, pred1_binary)print(f"Accuracy: {accuracy1:.4f}")print(f"\nConfusion Matrix:\n{cm1}")# Odds ratiosodds1 = pd.DataFrame(np.exp(log_model1.params), columns=['Odds Ratio'])odds1['p-value'] = log_model1.pvaluesodds1[['CI 2.5%', 'CI 97.5%']] = np.exp(log_model1.conf_int())print(f"\nOdds Ratios:\n{odds1}")

### 10.2 Predicting Mental Health Incident FlagWe model the probability of a mental health-related action at arrest based on year, sex, and location.

In [ ]:
# Model 2: Predict Mental Instability flagX2 = dataset_clean.drop(['Age_group__at_arrest_', 'StripSearch', 'Actions_at_arrest___Mental_inst'], axis=1)y2 = dataset_clean['Actions_at_arrest___Mental_inst']X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=0)formula2 = "Actions_at_arrest___Mental_inst ~ Arrest_Year + Sex + ArrestLocDiv"training2 = pd.concat([X2_train, y2_train], axis=1)log_model2 = smf.logit(formula2, data=training2).fit()print(log_model2.summary())

In [ ]:
# Model 2: Evaluationpred2 = log_model2.predict(X2_test)pred2_binary = list(map(round, pred2))accuracy2 = accuracy_score(y2_test, pred2_binary)cm2 = confusion_matrix(y2_test, pred2_binary)print(f"Accuracy: {accuracy2:.4f}")print(f"\nConfusion Matrix:\n{cm2}")# Odds ratiosodds2 = pd.DataFrame(np.exp(log_model2.params), columns=['Odds Ratio'])odds2['p-value'] = log_model2.pvaluesodds2[['CI 2.5%', 'CI 97.5%']] = np.exp(log_model2.conf_int())print(f"\nOdds Ratios:\n{odds2}")

## 11. Summary of Findings| Question | Result ||---|---|| Does gender affect strip searches? | **Yes** — Males significantly more likely to be strip searched || Does age group affect strip searches? | **No** — No significant main effect of age group alone || Has the practice changed over time? | **Yes** — Significant decline from 2020 to 2021 || Does gender affect mental health cases? | **Yes** — Males have significantly higher case counts || Which locations are hotspots? | **Division 51** dominates; Division 14 ranks second || Can we predict strip search likelihood? | **Yes** — Logistic regression achieves 88% accuracy || Can we predict mental health incidents? | **Yes** — Logistic regression achieves 96% accuracy |### Limitations- Unequal group sizes may affect ANOVA robustness- Only 2 years of data limits temporal trend analysis- Residual normality assumptions are violated in some ANCOVA models- A three-way ANOVA (Year × Sex × Age) could provide deeper insight